# Qwen2.5-3B-Instruct — Cybersecurity + Networking Fine-tune (v2, complete)

**What's new in this version:**
- Base model: **Qwen2.5-3B-Instruct** (we are not using the base, non-instruct model — the reason was discussed in chat)
- **All 4 datasets are combined into a single training run** (earlier sequential training caused short-answer/forgetting issues):
  1. `mariiazhiv/cybersecurity_qa`
  2. `jobby32/cybersecurity-QA-with-negatives` (new)
  3. `beyazitkelceoglu/networking_dataset_alpaca`
  4. `darkknight25/Networking_Commands_Dataset`
- `max_seq_length` increased to **4096** (room for long answers)
- Added a system prompt that encourages the model to give detailed answers
- **Diagnostic cell** that checks answer lengths BEFORE training — so the "short answers" problem gets caught early
- The schema of the `jobby32` dataset could not be confirmed (it is a new/niche dataset) — a cell prints it so you can inspect it; if auto-detect fails, add a manual format branch (the spot is marked)

**Run order**: top to bottom, one cell at a time.

## 1. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps --force-reinstall "git+https://github.com/unslothai/unsloth.git"


## 2. Load Qwen2.5-3B-**Instruct** (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096   # increased for long/detailed answers (was 2048 earlier)
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


## 3. LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 4. Load all 4 datasets and inspect columns
⚠️ The exact schema of `jobby32/cybersecurity-QA-with-negatives` could not be confirmed. Run this cell and
make sure to check the output of `print(jobby_ds[0])` — the next formatter cell depends on it.

In [ ]:
from datasets import load_dataset

cyber_ds1 = load_dataset("mariiazhiv/cybersecurity_qa", split="train")
jobby_ds  = load_dataset("jobby32/cybersecurity-QA-with-negatives", split="train")
net_ds1   = load_dataset("beyazitkelceoglu/networking_dataset_alpaca", split="train")
net_ds2   = load_dataset("darkknight25/Networking_Commands_Dataset", split="train")

print("cyber_ds1 columns:", cyber_ds1.column_names)
print("cyber_ds1 sample:", cyber_ds1[0])
print()
print("jobby_ds columns:", jobby_ds.column_names)
print("jobby_ds sample:", jobby_ds[0])
print()
print("net_ds1 columns:", net_ds1.column_names)
print("net_ds1 sample:", net_ds1[0])
print()
print("net_ds2 columns:", net_ds2.column_names)
print("net_ds2 sample:", net_ds2[0])


## 5. Chat template + universal formatter
A system prompt is added to push the model toward detailed answers
(but this only works when the training data answers are themselves detailed — the next diagnostic cell verifies this).

Several possible column patterns are tried for the `jobby32` dataset (`question`/`answer`,
`query`/`positive`, and `chosen`/`rejected`, which are common in "with-negatives" datasets).
If your printed output does not match any of these patterns, the error message will show the exact columns —
use them to add a new `elif` branch in `build_user_text`.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")

SYSTEM_PROMPT = (
    "You are an expert cybersecurity and networking assistant. "
    "Always answer with clear, detailed, technically accurate explanations. "
    "Avoid one-line answers; include relevant context, examples, or steps where useful."
)

def build_user_text(example, cols):
    # --- Alpaca-style (instruction/input/output) ---
    if "instruction" in cols and "output" in cols:
        instruction = example.get("instruction", "") or ""
        user_text = instruction
        assistant_text = example.get("output", "") or ""

    # --- Networking commands dataset: combine rich fields into a fuller answer ---
    elif "command" in cols and "description" in cols:
        command = example.get("command", "")
        description = example.get("description", "") or ""
        usage = example.get("usage", "") or ""
        category = example.get("category", "") or ""
        risk = example.get("risk", "") or ""
        user_text = f"Explain the networking command: {command}"
        assistant_text = (
            f"{description}\n\n"
            f"Usage: {usage}\n"
            f"Category: {category}\n"
            f"Risk/Notes: {risk}"
        ).strip()

    # --- Simple question/answer ---
    elif "question" in cols and "answer" in cols:
        user_text = example.get("question", "") or ""
        assistant_text = example.get("answer", "") or ""

    # --- "with-negatives" style: question + positive/chosen answer, ignore the negative ---
    elif "question" in cols and "positive" in cols:
        user_text = example.get("question", "") or ""
        assistant_text = example.get("positive", "") or ""
    elif "query" in cols and "positive" in cols:
        user_text = example.get("query", "") or ""
        assistant_text = example.get("positive", "") or ""
    elif "chosen" in cols and ("question" in cols or "prompt" in cols):
        q_col = "question" if "question" in cols else "prompt"
        user_text = example.get(q_col, "") or ""
        assistant_text = example.get("chosen", "") or ""
    elif "prompt" in cols and ("response" in cols or "completion" in cols):
        ans_col = "response" if "response" in cols else "completion"
        user_text = example.get("prompt", "") or ""
        assistant_text = example.get(ans_col, "") or ""
    elif "input" in cols and "output" in cols:
        user_text = example.get("input", "") or ""
        assistant_text = example.get("output", "") or ""
    else:
        raise ValueError(
            f"Could not detect the columns: {cols}. "
            f"Look at the output of print(jobby_ds[0]) above and add a new elif branch here."
        )
    return user_text.strip(), assistant_text.strip()

def format_dataset(dataset, use_system_prompt=True):
    cols = set(dataset.column_names)

    def formatting_func(examples):
        texts = []
        n = len(next(iter(examples.values())))
        for i in range(n):
            single_example = {k: examples[k][i] for k in examples}
            user_text, assistant_text = build_user_text(single_example, cols)
            convo = []
            if use_system_prompt:
                convo.append({"role": "system", "content": SYSTEM_PROMPT})
            convo.append({"role": "user", "content": user_text})
            convo.append({"role": "assistant", "content": assistant_text})
            texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
        return {"text": texts}

    return dataset.map(formatting_func, batched=True, remove_columns=dataset.column_names)

cyber_dataset1 = format_dataset(cyber_ds1)
jobby_dataset  = format_dataset(jobby_ds)
net_dataset1   = format_dataset(net_ds1)
net_dataset2   = format_dataset(net_ds2)

print(cyber_dataset1[0]["text"][:500])


## 6. Diagnostic — answer length check (IMPORTANT, do not skip this)
This cell tells you, BEFORE training, the average number of tokens in the answers of each dataset.
If a dataset's average is very low (~<25 tokens), that dataset will drive the final model's short-answer
behavior — fix it (as we did with darkknight25) and then move on.

In [ ]:
def avg_answer_tokens(dataset, name):
    lengths = []
    for row in dataset.select(range(min(100, len(dataset)))):
        # 'text' contains the full chat template; we approximate by counting tokens in the last (assistant) part
        assistant_part = row["text"].split("assistant")[-1]
        lengths.append(len(tokenizer.encode(assistant_part)))
    avg_len = sum(lengths) / len(lengths)
    print(f"{name}: avg ~{avg_len:.1f} tokens/answer (sample of {len(lengths)})")
    if avg_len < 25:
        print(f"  ⚠️  WARNING: {name} has very short answers — the final model will give short answers too!")
    return avg_len

avg_answer_tokens(cyber_dataset1, "cyber_dataset1 (mariiazhiv)")
avg_answer_tokens(jobby_dataset,  "jobby_dataset (jobby32)")
avg_answer_tokens(net_dataset1,   "net_dataset1 (beyazitkelceoglu)")
avg_answer_tokens(net_dataset2,   "net_dataset2 (darkknight25)")


## 7. Combine all 4 datasets into one training set

In [ ]:
from datasets import concatenate_datasets

combined_dataset = concatenate_datasets(
    [cyber_dataset1, jobby_dataset, net_dataset1, net_dataset2]
).shuffle(seed=3407)

print(combined_dataset)
print("\nTotal training examples:", len(combined_dataset))


## 8. Trainer setup

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,          # kept low (2 instead of 3) — to avoid overfitting to short templates
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_combined",
        report_to = "none",
    ),
)


## 9. GPU stats (optional)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


## 10. Train

In [ ]:
trainer_stats = trainer.train()


## 11. Save the fine-tuned model

In [ ]:
model.save_pretrained("qwen2.5-3b-cyber-networking-v2-lora")
tokenizer.save_pretrained("qwen2.5-3b-cyber-networking-v2-lora")

# To push to the HF Hub:
# model.push_to_hub("your-username/qwen2.5-3b-cyber-networking-v2-lora", token="hf_...")


## 12. Inference test — check for detailed answers

In [ ]:
FastLanguageModel.for_inference(model)

test_questions = [
    "What is a SQL injection attack and how can it be prevented?",
    "What does the 'nmap -sS' command do?",
    "Explain how a phishing attack typically works.",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=400, temperature=0.7, do_sample=True, use_cache=True
    )
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print(f"Q: {q}\nA: {response}\n{'-'*60}")


## 13. (Optional) Gradio app

In [ ]:
!pip install -q gradio


In [ ]:
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

def chat_fn(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        input_ids=inputs, streamer=streamer, max_new_tokens=512,
        temperature=0.7, top_p=0.9, do_sample=True, use_cache=True,
    )
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial_response = ""
    for new_text in streamer:
        partial_response += new_text
        yield partial_response

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Cybersecurity & Networking Assistant (Qwen2.5-3B fine-tuned)",
    description="Fine-tuned on cybersecurity_qa, cybersecurity-QA-with-negatives, and 2 networking datasets.",
    examples=[
        "What is a SQL injection attack?",
        "What does the 'traceroute' command do?",
        "Explain the CIA triad in cybersecurity.",
    ],
)
demo.launch(share=True, debug=True)


## 14. (Optional) Merge + export as GGUF for deployment

In [ ]:
# model.save_pretrained_merged("qwen2.5-3b-cyber-networking-v2-merged", tokenizer, save_method = "merged_16bit")
# model.save_pretrained_gguf("qwen2.5-3b-cyber-networking-v2-gguf", tokenizer, quantization_method = "q4_k_m")
